# Probability Flow ODE of Langevin Dynamics

Source: <BR>

https://mingxuan-yi.github.io/blog/2023/prob-flow-ode/

Adapted:

Antonio Esteves @UMinho, February 2025

---

This notebook provides a simple numerical approach using PyTorch to simulate the probability flow ordinary differential equation (ODE) of Langevin dynamics. This implementation slightly modifies the code of a Generative Adversarial Network (GAN), which can be considered to be a "non-parametric GAN" and an alternative perspective of the probability flow ODE.

Briefly speaking, we can remove the generator from the GAN and simulate a probability flow ODE similarly to a diffusion model.

In [ ]:
import torch
import torch.nn                   as     nn
import torch.nn.functional        as     F
import torch.optim                as     optim
from   torch.distributions.normal import Normal
import matplotlib.pyplot          as     plt
from   IPython.display            import display, clear_output
import numpy                      as     np
from   matplotlib.animation       import FuncAnimation, PillowWriter
from   matplotlib.collections     import LineCollection
from   tqdm                       import tqdm
from   sklearn                    import datasets

## 1. Example

We will use a modified version of the Neal’s funnel distribution. The funnel distribution is known to be difficult to sample from due to its irregular geometric properties. Langevin dynamics utilizes the log density information to explore the distribution but its particles fail to reach the bottom area of the funnel. The probability flow ODE succeeds in this task as it does not rely on the density function but on samples drawn from the target distribution.

Neal’s Funnel is an extreme distribution that exemplifies the difficulties of sampling from some hierarchical models. However, it can be reparameterized in such a way as to make sampling straightforward. Neal’s funnel has support for $y \in \mathbb{R}$ and $x \in \mathbb{R}^9$

$$
p(y,x) = \mathsf{normal}(y ; 0,3) * \prod_{n=1}^9 \mathsf{normal}(x_n ; 0,\exp(y/2))
$$

The probability contours are shaped like ten-dimensional funnels. The funnel’s neck is particularly sharp because of the exponential function applied to $y$. A plot of the log marginal density of $y$ and the first dimension $x_1$ is shown in the following plot.

![Log density plot of Neal's funnel distribution.](../fig/flow_ode_funnel_prob.png)

### Defining the funnel distribution

In [ ]:
class funnel():
    def __init__(self, y_std=1):
        self.y_dist = Normal(0, y_std)
    
    def log_prob_normal(self, mu, scale, x):
        cov = scale*scale
        a   = -0.5 * (np.log(2 * np.pi) + torch.log(cov))
        b   = -0.5 * (x - mu) * (x - mu) / cov
        return a + b
    
    def log_prob(self, data):
        x         = data[:, 0]
        y         = data[:, 1]
        y_log_pdf = self.y_dist.log_prob(y)
        x_log_pdf = self.log_prob_normal(0.0, self.x_std(y), x)
        return x_log_pdf + y_log_pdf
    
    def x_std(self, y):
        return torch.exp(y) / 10
    
    def sample(self, n_samples):
        y = self.y_dist.sample([n_samples])
        x = torch.randn_like(y) * self.x_std(y)
        return torch.vstack([x, y]).T 

### Visualizing the tunnel probability density function

In [ ]:
def plt_potential_func(log_prob, ax, num_grid=500, xs=[-5, 5], ys=[-5, 5], cmap=None):
 
    xlist  = torch.linspace(xs[0], xs[1], num_grid)
    ylist  = torch.linspace(ys[0], ys[1], num_grid)
    xx, yy = np.meshgrid(xlist, ylist)
    z      = np.hstack([xx.reshape(-1, 1), yy.reshape(-1, 1)])

    z      = torch.Tensor(z)
    u      = log_prob(z).cpu().numpy()
    p      = u.reshape(num_grid, num_grid)
    return ax.pcolor(xx, yy, p, cmap=cmap, vmin=-7.5, vmax=1.0)

funnel_dist = funnel()
fig, ax     = plt.subplots(figsize=(4, 4), frameon=False)
im          = plt_potential_func(funnel_dist.log_prob, ax)
ax.axis('equal')
ax.axis("off")
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
plt.margins(0,0)
display(fig)
plt.close()
fig.savefig('flow_ode_funnel_true.png', pad_inches = 0,transparent = False)

## 2. Langevin dynamics and its probability flow ODE

Langevin dynamics follows a stochastic differential equation (SDE) to describe the motion of a particle $\mathbf{x}_t \in \mathbb{R}^n$,

$$
\mathrm{d} \mathbf{x}_t = \nabla_\mathbf{x} \log p(\mathbf{x}_t)\mathrm{d}t + \sqrt{2} \mathrm{d}\mathbf{w}_t
$$

where $\mathbf{w}_t$ represents the Brownian motion. Using the Itô integration, we can obtain the Fokker-Placnk equation describing the marginal laws of the dynamics over time

$$
\frac{\partial q_t(\mathbf{x})}{\partial t} = \text{div}\Big[ q_t(\mathbf{x})\big(\nabla_\mathbf{x} \log q_t(\mathbf{x}) - \nabla_\mathbf{x} \log p(\mathbf{x}) \big) \Big]
$$

where $\text{div}$ is the [divergence operator](https://en.wikipedia.org/wiki/Divergence) in vector calculus.

If the target distribution decays at infinity $\lim_{\mathbf{x} \to \infty} p(\mathbf{x})= 0$, e.g., the Boltzmann distribution $p(\mathbf{x}) \propto \exp\big(-V(\mathbf{x})\big)$, the equilibrium (steady state) of the dynamics is achieved if and only if $q_t=p$ such that the infinitesimal change of the marginal is $\frac{\partial q_t}{\partial t}=0$. Evolving a particle from the initialization $\mathbf{x}_0 \sim q_0(\mathbf{x})$, its marginal $q_t(\mathbf{x})$ will eventually converge (weakly) to the stationary distribution $p(\mathbf{x})$. However, establishing the convergence rate within finite-time can be challenging; additional conditions for the target distribution must be met to guarantee convergence. For example, if $p(\mathbf{x})$ satisfies the [log-Sobolev inequality](https://en.wikipedia.org/wiki/Logarithmic_Sobolev_inequalities), then the marginal $q_t(\mathbf{x})$ will converge to $p(\mathbf{x})$ exponentially fast in terms of the Kullback-Leibler divergence. Nevertheless, we shall be able to expect that Langevin dynamics can at least find some local modes in practice.

In order to numerically simulate Langevin dynamics, we can use the Euler-Maruyama method to discretize the SDE, this gives the well-known **unadjusted Langevin algorithm (ULA)**

$$
\mathbf{x}_{i+1} \leftarrow \mathbf{x}_{i} + \epsilon \nabla_{\mathbf{x}} \log p(\mathbf{x}_i) + \sqrt{2\epsilon} \mathcal{N}(0, I), \quad i=0, 1, 2, \cdots
$$

Iteratively running ULA, we can gradually transport samples from $q_0(\mathbf{x})$ to $p(\mathbf{x})$ as shown in the previous demo to sample from the funnel distribution. ULA is widely used in large-scale machine learning. For example, it can be applied in the training of Bayesian neural networks and energy-based models, and it also serves as the sampling scheme for the earliest version of score-based diffusion models ([Song and Ermon, 2019](https://arxiv.org/abs/1907.05600)).

### Probability flow ODEs

There are several approaches to derive the probability flow ODE of Langevin dynamics. The most accessible one is using the result from [score-based diffusion models](https://yang-song.net/blog/2021/score/#probability-flow-ode). In score-based diffusion models, it is shown that each SDE has an associated probability flow ODE sharing the same marginal $q_t$ (also see the Eq. (13) in [Song et. al., 2021](https://arxiv.org/abs/2011.13456)). Simply using this result, we can convert the Langevin SDE to the associated ODE

$$
\mathrm{d} \mathbf{x}_t = \Big [\nabla_\mathbf{x} \log p(\mathbf{x}_t)- \nabla_\mathbf{x} \log q_t(\mathbf{x}_t)\Big]\mathrm{d}t
$$

The marginal laws of the probability flow ODE also follow the same Fokker-Planck equation. The probability flow ODE differs from Langevin dynamics only in the nature of particle evolution: the former is deterministic, while the latter is stochastic. Similarly, using the Euler method to discretize the ODE gives

$$
\mathbf{x}_{i+1} \leftarrow \mathbf{x}_{i} + \epsilon \big[\nabla_{\mathbf{x}} \log {p(\mathbf{x}_i)}- \nabla_{\mathbf{x}} \log {q_{i}({\mathbf{x}_i})}\big], \quad i=0, 1, 2, \cdots
$$

It is worth noting the vector field of the probability flow ODE is the gradient of the log density ratio $\log \big[p(\mathbf{x}_i) / q_{i}({\mathbf{x}_i})\big]$. If we have access to the log density ratio, we can use the Euler method to simulate the ODE. Next, we will discuss a method to obtain the log density ratio, analogous to training the discriminator in GANs.

### Estimating the vector field via the binary classification

Recall that in GANs, we train a discriminator to solve the following binary classification problem

$$
\max_D \qquad \mathbb{E}_{p} \big[\log D(\mathbf{x})\big]+ \mathbb{E}_{q_i} \big[\log (1-D(\mathbf{x}))\big]
$$

The optimal discriminator is given by (see Proposition 1 in [Goodfellow et. al. 2014](https://arxiv.org/abs/1406.2661))

$$
D^{*}(\mathbf{x}) = \frac{p(\mathbf{x})}{p(\mathbf{x}) + q_i(\mathbf{x})}
$$

Since the last layer of the discriminator $D^{*}(\mathbf{x})$ is activated by the Sigmoid function $\sigma(\cdot)$, inverting the Sigmoid activation gives the log density ratio

$$
\sigma^{-1}\big(D^{*}(\mathbf{x})\big) =\log \frac{p(\mathbf{x})}{q_{i}({\mathbf{x}})}
$$

where $\sigma^{-1}\big(D^{*}(\mathbf{x})\big)$ is called the logit output of a binary classifier.

This gives us a strategy for sampling via the probability flow ODE with bi-level optimization which is similar to training GANs,

*   Training the discriminator with a few steps of gradient update using samples from $q_i$ and $p$.
*   Computing the gradient of the logit output and updating particles using Eq. (1).

In the conventional GAN theory, training the discriminator was designed to estimate the Jensen-Shannon divergence (JSD), however, the smoothness between the estimated JSD and the generator’s distribution does not exist in practice because the divergence is reconstructed using samples. One can imagine that the Monte Carlo evaluation of the binary cross entropy involves a discrete step–sampling. This is the major reason causing the inconsistency between the GAN theory and its practical algorithms, e.g., the non-saturated tricks. The essential information we obtained in training the discriminator is not the JSD, but is the density ratio function which indicates the vector fields.

### Run Langevin dynamics

In [ ]:
def grad_logp(x, logp):
    '''
    Define the gradient calculation.
    '''
    return torch.autograd.grad(
        logp(x),
        x,
        grad_outputs = torch.ones_like(logp(x)), 
        create_graph = False
        )[0]

# Initialize 1024 particles with samples drawn from the Gaussian distribution, defined
# by two value (y,x_1), which we want to turn into samples from the funnel distribution
xs_langevin  = torch.randn([1024, 2])*0.5 + torch.tensor([.0, 2.0])

fig, ax      = plt.subplots(figsize=(4, 4))
scatter_plot = ax.scatter(xs_langevin[:, 0], xs_langevin[:, 1], s=5, alpha=0.2, color='red')
time_text    = ax.text(1.0, -3.5, '$t=0$', fontsize=14) 
ax.axis('equal')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.axis("off") # remove the axis will speed up the training
fig.tight_layout()

clear_output(wait=True)
display(fig)
xs_langevin.requires_grad=True
flows      = [xs_langevin.detach()]

n_iters    = 1500
lr         = 1e-5
decay_rate = 0.999

# Unadjusted Langevin dynamics step
for i in range(n_iters+1):
    xs_langevin = xs_langevin + 0.001*grad_logp(
        xs_langevin,
        funnel_dist.log_prob
        ) + np.sqrt(0.001*2) * torch.randn_like(xs_langevin)
    lr          = lr * decay_rate
    flows.append(xs_langevin.detach())
    scatter_plot.set_offsets(xs_langevin.detach())
    time_text.set_text(f'$t={i}/{n_iters}$')
    clear_output(wait=True)
    display(fig)
    print('[%d/%d]: lr, %.7f' % (i, n_iters, lr))
plt.close()


### Saving the animation

In [ ]:
def update(frame):
    scatter_plot.set_offsets(flows[frame])
    time_text.set_text(f'$t={frame}/{n_iters}$')

# Generate the GIF 
ani    = FuncAnimation(fig=fig, func=update, frames=tqdm(range(0, len(flows), 1)), interval=1)
writer = PillowWriter(fps=40)  
ani.save("flow_ode_funnel_langevin.gif", writer=writer)

## 3. Probability Flow ODE

### Building the model

In [ ]:
class Discriminator(nn.Module):
    '''
    Define an MLP discriminator.
    '''
    def __init__(self, d_input_dim=2):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(d_input_dim, 512)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features//2)
        self.fc3 = nn.Linear(self.fc2.out_features, 1)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.leaky_relu(self.fc2(x), 0.2)
        return self.fc3(x)

In [ ]:
# Create the funnel distribution <=> p(x)
funnel_dist = funnel()

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dim    = 2

# Loss is binary cross-entropy with logits
loss_fn     = nn.BCEWithLogitsLoss()

# Instantiate the discriminator
D_logits    = Discriminator().to(device)

# Initialize 1024 particles with samples drawn from the Gaussian distribution, defined
# by two value (y,x_1), which we want to turn into samples from the funnel distribution
xs          = (torch.randn([1024, 2])*0.5 + torch.tensor([0.0, 2.0])).to(device)
xs.requires_grad=True

# Select the discriminator optimizer
D_optimizer = optim.Adam(D_logits.parameters(), lr = 0.001)

# We can also use SGD optimizer for xs, which corresponds
# to the standard forward Euler approach, but it works very slow.

# Select the generator (particles) optimizer
G_optimizer = optim.SGD([xs], lr = 0.001)
# G_optimizer = optim.Adam([xs], lr = 0.001)

def train_one_step(batch_size=1024):
    D_losses, G_losses = [], []
    # Draw a batch of samples from the tunnel distribution -> x
    x = funnel_dist.sample(batch_size).to(device)

    D_optimizer.zero_grad() # reset discriminator gradients

    # Train the discriminator on x ~ p and labels y=1 (real samples)

    x_real, y_real = x.view(-1, data_dim), torch.ones(batch_size, 1).to(device)
    real_loss      = loss_fn(D_logits(x_real), y_real)

    # Train the discriminator xs ~ q_i(x) and labels y= (fake samples)

    # Draw a batch of indices from range (0:xs.shape[0])
    idx            = np.random.choice(xs.shape[0], batch_size, replace=False)
    # Draw a batch of samples from xs (current particles)
    x_fake, y_fake = xs[idx], torch.zeros(batch_size, 1).to(device)
    fake_loss      = loss_fn(D_logits(x_fake), y_fake)

    # Optimize only the discriminator weights
    D_loss = real_loss + fake_loss
    D_loss.backward()
    D_optimizer.step()

    G_optimizer.zero_grad() # reset generator gradients
    
    # Update the particles <=> update generator
    logit_xs = D_logits(xs)
    
    # Sum for all log ratios log(p(x)/q_i(x)) and backpropagate the sum, 
    # which gives the correct gradient for each particle
    G_loss = -torch.sum(logit_xs)
    G_loss.backward()
    G_optimizer.step()

    D_losses.append(D_loss.data.item())
    G_losses.append(G_loss.data.item())
    return torch.mean(torch.FloatTensor(D_losses)), torch.mean(torch.FloatTensor(G_losses))


### Simulation of the ODE

In [ ]:
n_iters     = 1500 
scheduler_G = torch.optim.lr_scheduler.ExponentialLR(optimizer=G_optimizer, gamma=0.9998)
scheduler_D = torch.optim.lr_scheduler.ExponentialLR(optimizer=D_optimizer, gamma=0.9998)
flows       = []

fig, ax      = plt.subplots(figsize=(4, 4))
scatter_plot = ax.scatter(xs.detach().cpu()[:, 0], xs.detach().cpu()[:, 1], s=5, alpha=0.2, color='red')
time_text    = ax.text(1.0, -3.5, '$t=0$', fontsize=14) 
ax.axis('equal')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.axis("off") # remove the axis will speed up the training
fig.tight_layout()

clear_output(wait=True)
display(fig)


for i in range(0, n_iters+1):
    D_loss, G_loss = train_one_step()
    particles_t    = xs.cpu().detach()
    scatter_plot.set_offsets(particles_t)
    time_text.set_text(f'$t={i}/{n_iters}$')
    flows.append(particles_t)
    clear_output(wait=True)
    display(fig)
    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f, D_lr: %.6f, G_lr: %.6f' % (
            i, n_iters, D_loss, G_loss, scheduler_D.get_last_lr()[0], scheduler_G.get_last_lr()[0]))
    scheduler_G.step()
    scheduler_D.step()
plt.close()

### Generating the animation

In [ ]:
def update(frame):
    scatter_plot.set_offsets(flows[frame])
    time_text.set_text(f'$t={frame}/{n_iters}$')

# Generate the GIF 
ani    = FuncAnimation(fig=fig, func=update, frames=tqdm(range(0, len(flows), 1)), interval=1)
writer = PillowWriter(fps=40)  
ani.save("flow_ode_funnel_ode.gif", writer=writer)  


## 4. Modifying the code of a GAN

* The first step is to remove the generator $G(z)$, instead we initialize particles `xs` and put them into the `SGD` optimizer corresponding to the Euler discretization of the ODE. We can also use `optim.Adam([xs], lr = 0.001)` to incorporate the momentum of gradients. Note that we use the loss function `nn.BCEWithLogitsLoss()`. This function directly works with the logit of the output of a binary classifier, where `D_logits` is a MLP in this example.

```Python
data_dim     = 2

# loss
criterion    = nn.BCEWithLogitsLoss()

# build the network
D_logits     = Discriminator().to(device)
-G           = Generator().to(device)
+xs          = make_8gaussians(n_samples=512).to(device)
+xs.requires_grad = True

# optimizer
D_optimizer  = optim.Adam(D_logits.parameters(), lr = 0.001)
-G_optimizer = optim.Adam(G.parameters(), lr = 0.001)
+G_optimizer = optim.SGD([xs], lr = 0.001) 
```

* The next step is to modify the training procedure of the GAN. There is no difference on training the discriminator, we just replace generated samples with a minibatch of `xs` and use a single step gradient descent to train the discriminator. The `G_loss` is now changed to `-torch.sum(D_logits(xs))` with the purpose of computing the gradient per particle. This is because there is no explicit way to perform batch gradient operation by backpropagating the loss in PyTorch. We can instead sum all per particle forward pass together to generate a scalar and backpropagating this scalar would give each particle its own gradient. An alternative is to use `torch.autograd` class but it more complicated.

```Python
#.................Train the discriminator ...............
D_optimizer.zero_grad() # eval real loss on p
x_real, y_real  = x.view(-1, data_dim), torch.ones(batch_size, 1).to(device)
real_loss       = criterion(D_logits(x_real), y_real)

# eval fake loss on q_i
-z              = torch.randn(batch_size, z_dim).to(device)
-x_fake, y_fake = G(z), torch.zeros(batch_size, 1).to(device)
# randomly select some particles to train the discriminator
+idx            = np.random.choice(xs.shape[0], batch_size, replace=False)

# Gradient backpropagation and optimize discriminator parameters
+x_fake, y_fake = xs[idx], torch.zeros(batch_size, 1).to(device)
fake_loss       = criterion(D_logits(x_fake), y_fake)
D_loss          = real_loss + fake_loss
D_loss.backward()
D_optimizer.step()

#.............. Update particles .................
G_optimizer.zero_grad()
-z              = torch.randn(batch_size, z_dim).to(device)
-x_fake, y_fake = G(z), torch.ones(batch_size, 1).to(device)
-G_loss         = criterion(D_logits(x_fake), y_fake) # non-saturate trick
# update all particles
# allow for the batch gradient for each particle
+G_loss         = -torch.sum(D_logits(xs))
G_loss.backward()
G_optimizer.step() 
```

Now we can run the code to simulate the probability flow ODE.

![Using the SGD optimizer.](../fig/flow_ode_s_sgd.gif)

![Using the Adam optimizer.](../fig/flow_ode_s_adam.gif)


### Visualizing the data

In [ ]:
def make_moon_data(n_samples, noise=0.08):
    data, _     = datasets.make_moons(n_samples=n_samples, noise=noise) 
    data[:, 0] += -0.5
    return torch.tensor(data.astype("float32"))

def make_s_curve_data(n_samples, noise=0.12):
    data, _ = datasets.make_s_curve(n_samples=n_samples, noise=noise)
    data    = data[:, [0, 2]] / 2.0
    return torch.tensor(data.astype("float32"))

def make_8gaussians(n_samples, scale=7, magnitude=0.25):
    # modified from https://github.com/wgrathwohl/LSD.git
    centers = [
        ( 1,  0), 
        (-1,  0), 
        ( 0,  1), 
        ( 0, -1), 
        ( 1. / np.sqrt(2),  1. / np.sqrt(2)),
        ( 1. / np.sqrt(2), -1. / np.sqrt(2)),
        (-1. / np.sqrt(2),  1. / np.sqrt(2)), 
        (-1. / np.sqrt(2), -1. / np.sqrt(2))
        ]
    centers = [
        (scale * x, scale * y) for x, y in centers
        ]

    dataset = []
    for i in range(n_samples):
        point     = np.random.randn(2) * 0.5
        idx       = np.random.randint(8)
        center    = centers[idx]
        point[0] += center[0]
        point[1] += center[1]
        dataset.append(point)
    dataset  = np.array(dataset, dtype="float32")
    dataset *= magnitude
    return torch.tensor(dataset)

# choose the moon data or the s_curve data
#data = make_moon_data(n_samples=500)
data      = make_s_curve_data(n_samples=512)
particles = make_8gaussians(n_samples=512)
fig, ax   = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0], data[:, 1], s=5, alpha=0.2, label=r'Target')
scatter_plot = ax.scatter(particles[:, 0], particles[:, 1], s=5, alpha=0.5, label=r'Particles', color='red')
ax.axis('equal')
ax.axis("off") # remove the axis will speed up the training
ax.legend(loc='upper right', fontsize=15, markerscale=2)
fig.tight_layout()
plt.show()

### Building the model

In [ ]:
# define a MLP discriminator
class Discriminator(nn.Module):
    def __init__(self, d_input_dim=2):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(d_input_dim, 512)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features//2)
        self.fc3 = nn.Linear(self.fc2.out_features, 1)

    # forward method
    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.leaky_relu(self.fc2(x), 0.2)
        return self.fc3(x)

In [ ]:
# build network
data_dim  = 2

# loss
criterion = nn.BCEWithLogitsLoss()

D_logits  = Discriminator().to(device)
xs        = make_8gaussians(n_samples=512).to(device)
xs.requires_grad=True

# optimizer
D_optimizer = optim.Adam(D_logits.parameters(), lr = 0.001)

'''
One can also use SGD optimizer for 'xs', which corresponds
to the standard forward Euler approach, but it works very slow.
'''
G_optimizer = optim.SGD([xs], lr = 0.001)
#G_optimizer = optim.Adam([xs], lr = 0.001)

def train_per_step(batch_size=512):
    D_losses, G_losses = [], []
    x                  = make_s_curve_data(batch_size).to(device)

    #==============Train the discriminator===============#
    D_optimizer.zero_grad()
    # train discriminator on p
    x_real, y_real = x.view(-1, data_dim), torch.ones(batch_size, 1).to(device)
    real_loss      = criterion(D_logits(x_real), y_real)

    # train discriminator on q_i
    idx            = np.random.choice(xs.shape[0], batch_size, replace=False)
    x_fake, y_fake = xs[idx], torch.zeros(batch_size, 1).to(device)
    fake_loss      = criterion(D_logits(x_fake), y_fake)

    # gradient backprop & optimize ONLY D's parameters
    D_loss = real_loss + fake_loss
    D_loss.backward()
    D_optimizer.step()

    #==============Update particles===============#
    G_optimizer.zero_grad()
    
    # update all particles
    # batch gradient for each particle
    ''' torch.sum(logit_xs) is the sum for all log ratios,
            back-propagating the sum returns the gradient for each particle.
    '''
    G_loss = -torch.sum(D_logits(xs))
    G_loss.backward()
    G_optimizer.step()

    D_losses.append(D_loss.data.item())
    G_losses.append(G_loss.data.item())
    return torch.mean(torch.FloatTensor(D_losses)), torch.mean(torch.FloatTensor(G_losses))

### Simulation of the ODE

In [ ]:
n_iters     = 2500
scheduler_G = torch.optim.lr_scheduler.ExponentialLR(optimizer=G_optimizer, gamma=0.9998)
scheduler_D = torch.optim.lr_scheduler.ExponentialLR(optimizer=D_optimizer, gamma=0.9998)
flows       = [xs.cpu().detach()]
clear_output(wait=True)
scatter_plot.set_offsets(xs.cpu().detach())

for k in range(0, n_iters):
    D_loss, G_loss = train_per_step()
    particles_t    = xs.cpu().detach()
    scatter_plot.set_offsets(particles_t)
    flows.append(particles_t)
    clear_output(wait=True)
    display(fig)
    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f, D_lr: %.6f, G_lr: %.6f' % (
            k, n_iters, D_loss, G_loss, scheduler_D.get_last_lr()[0], scheduler_G.get_last_lr()[0]))
    scheduler_G.step()
    scheduler_D.step()
flows = torch.stack(flows)
plt.close()

### Generating the animation

In [ ]:
ax.scatter(flows[0][:, 0], flows[0][:, 1], s=5, alpha=0.2, color='black', zorder=2)
lines = LineCollection([], lw=1.0, alpha=0.15, color='darkgrey', zorder=0)
ax.add_collection(lines)

def update(frame):
    scatter_plot.set_offsets(flows[frame])
    lines.set_segments(flows[0:frame].permute(1,0,2))
    
# Generate the GIF 
ani    = FuncAnimation(fig=fig, func=update, frames=tqdm(range(0, len(flows), 3)), interval=1)
writer = PillowWriter(fps=50)  
ani.save("flow_ode_s_sgd_generated.gif", writer=writer)  